# Block 4:FAISS Vector Store 

### What is FAISS?

FAISS = Facebook AI Similarity Search

Think of it as a:
"super fast vector search engine"

Normal search:  finds exact keyword matches
FAISS search:   finds meaning matches ✅

Example:
You search → "memory problems in elderly"
FAISS finds → "cognitive decline in aging patients"
Same meaning, different words! 🎯

###  Load the Embeddings

 Understand before running:

np.load()        → loads our saved vectors back

embeddings.shape → should show (10, 384)i,e.,10 chunks, 384 numbers each

We need BOTH:

embeddings → to build searchable index

chunks     → to return text when searching

In [2]:
# Cell 1: Load embeddings and chunks we saved in Block 3
# embeddings.npy → our vectors
# chunks.json    → our original text

import numpy as np
import json

# Load vectors
embeddings = np.load("../data/processed/embeddings.npy")

# Load chunks
with open("../data/processed/chunks.json", "r") as f:
    chunks = json.load(f)

print(f"Embeddings shape: {embeddings.shape}")
print(f"Chunks loaded: {len(chunks)}")
print(f"\nReady to build FAISS index!")

Embeddings shape: (10, 384)
Chunks loaded: 10

Ready to build FAISS index!


###  Build The FAISS Index

In [3]:
# Cell 2: Build FAISS index
# This creates a searchable vector database

import faiss

# Get vector dimensions (384)
dimension = embeddings.shape[1]

# Create FAISS index
# IndexFlatL2 = exact search using L2 distance
index = faiss.IndexFlatL2(dimension)

# Convert embeddings to float32 (FAISS requirement)
embeddings_float32 = embeddings.astype(np.float32)

# Add vectors to index
index.add(embeddings_float32)

print(f"FAISS index built successfully!")
print(f"Vector dimension: {dimension}")
print(f"Vectors stored: {index.ntotal}")

FAISS index built successfully!
Vector dimension: 384
Vectors stored: 10


### Test The Search!
This is the most exciting cell so far — watch FAISS find relevant papers by meaning!

In [4]:
# Cell 3: Search the FAISS index!
# Convert question to vector → find similar chunks

from sentence_transformers import SentenceTransformer

# Load same embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Our test question
question = "What blood tests can detect Alzheimer's early?"

# Convert question to vector
question_vector = model.encode([question])
question_vector = question_vector.astype(np.float32)

# Search FAISS for top 3 similar chunks
distances, indices = index.search(question_vector, k=3)

print(f"Question: {question}")
print(f"\nTop 3 most relevant chunks:\n")

for rank, (dist, idx) in enumerate(zip(distances[0], indices[0])):
    print(f"Rank {rank+1}:")
    print(f"  Title: {chunks[idx]['title']}")
    print(f"  Distance: {dist:.4f}")
    print(f"  Text: {chunks[idx]['text'][:150]}...")
    print()

c:\Users\tpriy\DatasciencePortfolio\alzheimers-early-detection-rag-chatbot\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 685.31it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Question: What blood tests can detect Alzheimer's early?

Top 3 most relevant chunks:

Rank 1:
  Title: Plasma p-tau217, p-tau181 and Aβ42/40 for Alzheimer's disease diagnosis: ROC accuracy and 18F-florbetapir amyloid PET-CT concordance.
  Distance: 0.9378
  Text: Plasma p-tau217, p-tau181 and Aβ42 40 for Alzheimer s disease diagnosis: ROC accuracy and 18F-florbetapir amyloid PET-CT concordance.. Early diagnosis...

Rank 2:
  Title: Tears as a window to Alzheimer's disease: A systematic review of biomarkers for early detection.
  Distance: 0.9418
  Text: Tears as a window to Alzheimer s disease: A systematic review of biomarkers for early detection.. Alzheimer s disease (AD) is the leading cause of dem...

Rank 3:
  Title: Progression of fiber bundle damage in amnestic Alzheimer's disease and LATE: a 2-year fixel-based study.
  Distance: 1.0420
  Text: Progression of fiber bundle damage in amnestic Alzheimer s disease and LATE: a 2-year fixel-based study.. Alzheimer s disease (AD) and 

### Save The FAISS Index

In [1]:
# Cell 4: Save FAISS index to disk
# So we don't rebuild it every time

import os

os.makedirs("../vector_store/faiss_index", exist_ok=True)

# Save FAISS index
faiss.write_index(index, "../vector_store/faiss_index/alzheimer.index")

print("FAISS index saved successfully!")
print("Location: vector_store/faiss_index/alzheimer.index")

NameError: name 'faiss' is not defined